# Food Delivery Business Performance — Visualization Portfolio

**Day 14 Assignment**

This notebook uses the Food Delivery Business Performance dataset to explore business performance through **Matplotlib and Seaborn**. It covers trends in orders and revenue, city and cuisine performance, marketing effectiveness, delivery/customer experience, weather effects, order channels, and correlations.

### Dataset
- **Rows:** 360
- **Columns:** 16
- **Date range:** 2025-01-02 to 2025-12-28
- **Missing values:** 0
- **Duplicate rows after cleaning:** 0

> **Note:** Correlation shows association, not causation. Business conclusions should therefore be interpreted as patterns in the observed data rather than proof that one variable causes another.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv("Day14_Food_Delivery_Visualization_Dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])

numeric_cols = [
    "Orders", "Average_Order_Value", "Revenue", "Marketing_Spend",
    "Discounts", "Avg_Delivery_Minutes", "Customer_Rating",
    "Repeat_Customer_Percent"
]

df.info()
display(df.head())
display(df.describe().round(2))


## 1. Data Quality Check
The dataset is checked for missing values, duplicates, data types, and basic numerical ranges before visualization.

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing_Count"))

print("Duplicate rows:", df.duplicated().sum())

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))


## 2. Line Plot — Orders and Revenue Over Time
**Question:** How do order volume and revenue change across the year?

**Insight:** The monthly aggregation reveals the overall business trend and helps identify stronger and weaker periods. Revenue and orders generally move together because more orders contribute directly to total sales.

In [ ]:
monthly = (
    df.groupby(df["Date"].dt.to_period("M"))
      .agg(Orders=("Orders", "sum"), Revenue=("Revenue", "sum"))
      .reset_index()
)
monthly["Date"] = monthly["Date"].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly["Date"], monthly["Orders"], marker="o", label="Orders")
ax1.set_xlabel("Month")
ax1.set_ylabel("Total Orders")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly["Date"], monthly["Revenue"], marker="s", label="Revenue")
ax2.set_ylabel("Revenue (₹)")

plt.title("Monthly Orders and Revenue Trend")
plt.tight_layout()
plt.show()

display(monthly.round(2))


## 3. Bar Chart — City Performance
**Question:** Which cities generate the most orders and revenue?

**Insight:** **Bengaluru** is the strongest city by total revenue, followed closely by **Delhi**. This suggests these markets are important contributors to overall business performance and may deserve continued operational and marketing attention.

In [ ]:
city = (
    df.groupby("City")
      .agg(Orders=("Orders","sum"),
           Revenue=("Revenue","sum"),
           Avg_Rating=("Customer_Rating","mean"),
           Avg_Delivery=("Avg_Delivery_Minutes","mean"))
      .sort_values("Revenue", ascending=False)
)

plt.figure(figsize=(11, 5))
sns.barplot(data=city.reset_index(), x="City", y="Revenue")
plt.title("Revenue by City")
plt.xlabel("City")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(city.round(2))


## 4. Bar Chart — Cuisine Performance
**Question:** Which cuisines contribute most to revenue?

**Insight:** **Healthy cuisine** produces the highest total revenue in the dataset, followed by North Indian and Biryani. The result suggests that demand for healthier food choices is commercially significant in this sample.

In [ ]:
cuisine = (
    df.groupby("Cuisine")
      .agg(Orders=("Orders","sum"),
           Revenue=("Revenue","sum"),
           Avg_Rating=("Customer_Rating","mean"))
      .sort_values("Revenue", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=cuisine.reset_index(), x="Cuisine", y="Revenue")
plt.title("Revenue by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

display(cuisine.round(2))


## 5. Scatter Plot — Marketing Spend vs Revenue
**Question:** Is higher marketing spend associated with higher revenue?

**Insight:** The correlation between marketing spend and revenue is approximately **0.20**, indicating a **weak positive relationship**. In this dataset, larger marketing budgets are associated with somewhat higher revenue, but marketing spend alone does not explain most revenue variation.


In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=df, x="Marketing_Spend", y="Revenue", scatter_kws={"alpha": 0.6})
plt.title("Marketing Spend vs Revenue")
plt.xlabel("Marketing Spend (₹)")
plt.ylabel("Revenue (₹)")
plt.tight_layout()
plt.show()

print("Pearson correlation:", round(df["Marketing_Spend"].corr(df["Revenue"]), 3))


## 6. Scatter Plot — Delivery Time vs Customer Rating
**Question:** Does slower delivery correspond to lower customer ratings?

**Insight:** The correlation between delivery time and customer rating is **-0.88**, showing a **strong negative relationship**. Longer delivery times are associated with lower ratings, making delivery speed an important customer-experience metric.


In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=df, x="Avg_Delivery_Minutes", y="Customer_Rating",
            scatter_kws={"alpha": 0.6})
plt.title("Delivery Time vs Customer Rating")
plt.xlabel("Average Delivery Time (minutes)")
plt.ylabel("Customer Rating")
plt.tight_layout()
plt.show()

print("Pearson correlation:", round(
    df["Avg_Delivery_Minutes"].corr(df["Customer_Rating"]), 3
))


## 7. Histogram — Distribution of Delivery Times
**Question:** What delivery times are most common?

**Insight:** Most observations cluster around the central delivery-time range, while a smaller number of batches have considerably longer delivery times. These slower cases are potential operational outliers worth investigating.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="Avg_Delivery_Minutes", bins=20, kde=True)
plt.title("Distribution of Average Delivery Times")
plt.xlabel("Average Delivery Time (minutes)")
plt.ylabel("Number of Order Batches")
plt.tight_layout()
plt.show()


## 8. Box Plot — Delivery Time by Weather
**Question:** Does weather affect delivery time?

**Insight:** Rainy conditions have visibly higher delivery times than clear conditions, indicating that weather can create operational delays.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Weather", y="Avg_Delivery_Minutes")
plt.title("Delivery Time by Weather Condition")
plt.xlabel("Weather")
plt.ylabel("Average Delivery Time (minutes)")
plt.tight_layout()
plt.show()


## 9. Violin Plot — Customer Rating by Weather
**Question:** How does the distribution of customer ratings vary by weather?

**Insight:** Clear-weather batches have a higher average rating (**4.53**) than rainy batches (**4.23**). Rainy periods also show more spread toward lower ratings, consistent with the longer delivery times observed above.


In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(data=df, x="Weather", y="Customer_Rating", inner="box")
plt.title("Customer Rating Distribution by Weather")
plt.xlabel("Weather")
plt.ylabel("Customer Rating")
plt.tight_layout()
plt.show()


## 10. Count Plot — Order Channel
**Question:** Which ordering channels are used most frequently?

**Insight:** The **App** is by far the most common order channel in the dataset. This indicates that the app is the primary customer touchpoint and should remain a major focus for user experience and retention initiatives.

In [ ]:
channel = (
    df.groupby("Order_Channel")
      .agg(Orders=("Orders","sum"),
           Revenue=("Revenue","sum"),
           Avg_Rating=("Customer_Rating","mean"))
      .sort_values("Orders", ascending=False)
)

plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="Order_Channel", order=df["Order_Channel"].value_counts().index)
plt.title("Number of Order Batches by Order Channel")
plt.xlabel("Order Channel")
plt.ylabel("Number of Batches")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(channel.round(2))


## 11. Bar Chart — Revenue by Order Channel
**Question:** Does the dominant channel also generate the most revenue?

**Insight:** The App leads not only in usage but also in total revenue, reinforcing its importance as the main sales channel.

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=channel.reset_index(), x="Order_Channel", y="Revenue")
plt.title("Revenue by Order Channel")
plt.xlabel("Order Channel")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 12. Weather Impact on Business Performance
**Question:** What happens to delivery time, ratings, orders and revenue under different weather conditions?

**Insight:** Rainy weather has the highest average delivery time at **38.75 minutes**, compared with **29.99 minutes** in clear weather. Rainy periods also have lower average ratings. This suggests that weather-related operational planning can help protect customer satisfaction.


In [ ]:
weather = (
    df.groupby("Weather")
      .agg(Orders=("Orders","sum"),
           Revenue=("Revenue","sum"),
           Avg_Delivery=("Avg_Delivery_Minutes","mean"),
           Avg_Rating=("Customer_Rating","mean"))
      .sort_values("Revenue", ascending=False)
)

display(weather.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=weather.reset_index(), x="Weather", y="Avg_Delivery", ax=ax)
ax.set_title("Average Delivery Time by Weather")
ax.set_xlabel("Weather")
ax.set_ylabel("Average Delivery Time (minutes)")
plt.tight_layout()
plt.show()


## 13. Correlation Heatmap
**Question:** Which numerical variables move together?

**Insight:** Orders and revenue show a strong positive association because higher order volume contributes to revenue. Delivery time and customer rating have a strong negative relationship, while customer rating and repeat-customer percentage are strongly positive. Marketing spend has only a modest positive relationship with revenue.

In [ ]:
numeric_cols = [
    "Orders", "Average_Order_Value", "Revenue", "Marketing_Spend",
    "Discounts", "Avg_Delivery_Minutes", "Customer_Rating",
    "Repeat_Customer_Percent"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Business Performance Metrics")
plt.tight_layout()
plt.show()

display(corr.round(2))


# Final Summary of Key Findings

1. **City performance:** Bengaluru is the highest-revenue city in the dataset, with Delhi also performing strongly.
2. **Cuisine performance:** Healthy generates the most revenue among the listed cuisines.
3. **Marketing effectiveness:** Marketing spend and revenue have a weak positive correlation of about **0.20**, so revenue depends on more than marketing expenditure alone.
4. **Delivery and satisfaction:** Delivery time and customer rating have a strong negative correlation of about **-0.88**. Faster delivery is associated with better ratings.
5. **Weather impact:** Rainy conditions increase average delivery time and are associated with lower customer ratings than clear conditions.
6. **Order channels:** The **App** is the dominant channel and also generates the highest revenue.
7. **Customer retention:** Customer rating is positively associated with repeat-customer percentage, suggesting that better customer experiences may coincide with stronger repeat behavior.
8. **Business implication:** The strongest opportunities are to protect delivery speed during poor weather, continue investing in the app experience, and use city/cuisine-level performance to target marketing and operational resources.

### Conclusion
The visualization portfolio shows that food-delivery performance is shaped by a combination of **order volume, location, cuisine, channel, marketing, delivery operations, and external conditions such as weather**. The most actionable pattern is the relationship between delivery time and customer satisfaction: operational improvements that reduce delays can potentially improve the customer experience and repeat business.


## Submission Checklist

- [x] Matplotlib visualizations
- [x] Seaborn visualizations
- [x] Line plot
- [x] Bar charts
- [x] Scatter plots
- [x] Histogram
- [x] Box plot
- [x] Violin plot
- [x] Count plot
- [x] Correlation heatmap
- [x] Titles and axis labels
- [x] Interpretations for each visualization
- [x] Final summary of important findings
- [x] Dataset cleaning/validation
